# 07 - Versioning, Testing Strategy, and CLI Operations

## Definition
Model versioning tracks lineage of artifacts and enables safe promotion/rollback.

## Theory
Registry metadata (metrics, hashes, parent version) provides auditability.

## Motivation
When production quality degrades, rollback must be explicit and fast.


In [ ]:
from pathlib import Path
import os

CWD = Path.cwd()
ROOT = CWD if (CWD / "pyproject.toml").exists() else CWD.parent
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/benchmarks").mkdir(parents=True, exist_ok=True)


In [ ]:
import json
from pathlib import Path

registry_path = Path("models/registry.json")
registry = json.loads(registry_path.read_text(encoding="utf-8"))
registry


## Version Story in This Project
- `v1`: LogisticRegression baseline
- `v2`: benchmark-selected improved model

### Best Practice
Always keep parent-child linkage for traceability.


In [ ]:
from ml_package.versioning import VersionRegistry

tmp_registry = Path("outputs/benchmarks/notebook07_registry.json")
if tmp_registry.exists():
    tmp_registry.unlink()

vr = VersionRegistry(str(tmp_registry))
vr.register("v1", "models/iris_model_v1.pkl", metrics={"f1_macro": 0.96}, description="baseline", allow_overwrite=True)
vr.activate("v1")
vr.register("v2", "models/iris_model_v2.pkl", metrics={"f1_macro": 1.00}, description="improved", parent_version="v1", allow_overwrite=True)
vr.activate("v2")
vr.rollback_to("v1")
vr.list_versions()


## Testing Framework

### Unit Tests
Validation, loader, prediction engine, version registry.

### Integration Tests
FastAPI routes through ASGI + HTTP client.

### API Live Tests
`requests` tests included with sandbox-aware skip when sockets are blocked.


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_versioning.py", "tests/test_api.py"],
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
print("Exit code:", result.returncode)


## CLI Interface Demonstration
CLI is useful for batch scoring and automation where full API server is unnecessary.


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "ml_package.cli.predict",
    "--model-path",
    "models/iris_model.pkl",
    "predict",
    "--sepal-length",
    "5.1",
    "--sepal-width",
    "3.5",
    "--petal-length",
    "1.4",
    "--petal-width",
    "0.2",
]
out = subprocess.run(cmd, capture_output=True, text=True, check=False)
print(out.stdout)
print("CLI exit code:", out.returncode)
